In [ ]:
!pip install llama-index llama-index-embeddings-openai llama-index-llms-openai

In [ ]:
!pip install llama-index-vector-stores-chroma

In [ ]:
import os
import nest_asyncio
from getpass import getpass
from dotenv import load_dotenv

load_dotenv(r"C:\Users\anteb\PycharmProjects\JupyterProject\.env")
nest_asyncio.apply()

# Debug: Print environment variables to verify they are loaded
print("CO_API_KEY:", os.environ.get('CO_API_KEY'))
print("OPENAI_API_KEY:", os.environ.get('OPENAI_API_KEY'))
print("QDRANT_URL:", os.environ.get('QDRANT_URL'))
print("QDRANT_API_KEY:", os.environ.get('QDRANT_API_KEY'))

CO_API_KEY = os.environ.get('CO_API_KEY') or getpass("Enter CO_API_KEY: ")
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY') or getpass("Enter OPENAI_API_KEY: ")
QDRANT_URL = os.environ.get('QDRANT_URL') or getpass("Enter QDRANT_URL: ")
QDRANT_API_KEY = os.environ.get('QDRANT_API_KEY') or getpass("Enter QDRANT_API_KEY: ")

In [ ]:
from llama_index.core.settings import Settings
from llama_index.llms.cohere import Cohere
from llama_index.embeddings.openai import OpenAIEmbedding

from llama_index.llms.openai import OpenAI

Settings.llm = OpenAI(api_key=OPENAI_API_KEY, model_name="gpt-4o-mini")
Settings.embed_model = OpenAIEmbedding(model_name="text-embedding-3-small")

from llama_index.core import SimpleDirectoryReader

# Load your documents
documents = SimpleDirectoryReader(
    input_files=[r"C:\Users\anteb\PycharmProjects\JupyterProject\naive_rag\data\raw\2502.20364.pdf"],
    filename_as_id=True
).load_data()



In [ ]:
print(documents[3])

In [ ]:
!pip install chromadb

In [ ]:

from naive_rag.helpers.IngestionCacheManager import SmartIngestionCache
from chromadb import Client
from chromadb.config import Settings as ChromaSettings
from llama_index.core.ingestion import IngestionCache, IngestionPipeline
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.vector_stores.chromadb import ChromaVectorStore
from llama_index.core import VectorStoreIndex

# Initialize the Chroma client with a persist directory
chroma_client = Client(
    settings=ChromaSettings(
        persist_directory="./chroma_db",  # Adjust the directory as needed
    )
)

# Create the vector store using Chroma
vector_store = ChromaVectorStore(
    client=chroma_client,
    collection_name="rag_articles"
)

# Get your ingestion cache
ingest_cache = SmartIngestionCache().get_cache()

# Create the ingestion pipeline with text splitting and embedding transformations
pipeline = IngestionPipeline(
    transformations=[
        TokenTextSplitter(chunk_size=256, chunk_overlap=16),
        Settings.embed_model
    ],
    docstore=SimpleDocumentStore(),
    vector_store=vector_store,
    cache=ingest_cache,
)

# Run the pipeline to process the documents
nodes = pipeline.run(documents=documents)

# Build the vector store index from Chroma
index = VectorStoreIndex.from_vector_store(vector_store=vector_store)